In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import datetime, timedelta
import random

# Esquema da tabela bronze
clientes_schema = StructType([
    StructField('cliente_id', IntegerType(), False),
    StructField('nome', StringType(), True),
    StructField('email', StringType(), True),
    StructField('data_nascimento', DateType(), True),
    StructField('pais', StringType(), True)
])

# Gerar dados fake
nomes = ['Ana', 'Bruno', 'Carlos', 'Daniela', 'Eduardo', 'Fernanda', 'Gabriel', 'Helena', 'Igor', 'Juliana',
         'Kleber', 'Larissa', 'Marcos', 'Natália', 'Otávio', 'Patrícia', 'Quintino', 'Renata', 'Samuel', 'Tatiane',
         'Ulisses', 'Valéria', 'Wagner', 'Xuxa', 'Yasmin']
paises = ['Brasil', 'Argentina', 'Chile', 'Uruguai', 'Paraguai']

clientes_data = []
for i in range(1, 26):
    nome = nomes[i-1]
    email = f"{nome.lower()}@exemplo.com"
    data_nascimento = datetime(1980, 1, 1) + timedelta(days=random.randint(0, 15000))
    pais = random.choice(paises)
    clientes_data.append((i, nome, email, data_nascimento, pais))

clientes_df = spark.createDataFrame(clientes_data, schema=clientes_schema)

# Salvar como tabela bronze
clientes_df.write.mode("overwrite").saveAsTable("bronze.clientes")

display(clientes_df)

In [0]:
from pyspark.sql.functions import rand, round, col
clientes_df.write \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("bronze.clientes")

display(clientes_df)

# Adiciona coluna 'renda' (entre 2000 e 20000) e 'investimento' (entre 0 e 100000)
clientes_df = clientes_df.withColumn(
    "renda",
    round(rand() * 18000 + 2000, 2)
).withColumn(
    "investimento",
    round(rand() * 100000, 2)
)

# Atualiza a tabela bronze
clientes_df.write.mode("overwrite").saveAsTable("bronze.clientes")



In [0]:
from pyspark.sql.functions import *

In [0]:
clientes_df = clientes_df.withColumn(
    "seg_principal",
    when(
        (col("investimento") > 40000) & (col("renda") > 5000),
        "principal"
    ).otherwise("outros")
)

display(clientes_df)

In [0]:
clientes_df = clientes_df.drop("segmento")
